In [0]:
-- ============================================
-- 1. CREATE CATALOG & SCHEMA
-- ============================================

CREATE CATALOG IF NOT EXISTS training_catalog;

CREATE SCHEMA IF NOT EXISTS training_catalog.sql_demo;

USE CATALOG training_catalog;
USE SCHEMA sql_demo;

-- ============================================
-- 2. CREATE EMPLOYEES TABLE (DELTA)
-- ============================================

CREATE OR REPLACE TABLE employees
(
    emp_id INT,
    emp_name STRING,
    department STRING,
    salary INT,
    joining_date DATE
)
USING DELTA;

INSERT INTO employees VALUES
(101,'Pooja','IT',50000,'2026-07-01'),
(102,'Rahul','HR',60000,'2026-06-15'),
(103,'Anita','Finance',55000,'2026-05-10'),
(104,'John','IT',65000,'2026-07-18'),
(105,'Priya','Sales',45000,'2026-04-20');

SELECT * FROM employees;

-- ============================================
-- 3. CREATE DEPARTMENTS TABLE
-- ============================================

CREATE OR REPLACE TABLE departments
(
    dept_name STRING
)
USING DELTA;

INSERT INTO departments VALUES
('IT'),
('HR'),
('Finance');

SELECT * FROM departments;

-- ============================================
-- 4. SEMI JOIN
-- Employees whose department exists
-- ============================================

SELECT *
FROM employees
SEMI JOIN departments
ON employees.department = departments.dept_name;

-- ============================================
-- 5. ANTI JOIN
-- Employees whose department doesn't exist
-- ============================================

SELECT *
FROM employees
ANTI JOIN departments
ON employees.department = departments.dept_name;

-- ============================================
-- 6. DATE FUNCTIONS
-- ============================================

SELECT
CURRENT_DATE() AS today,
DATE_ADD(CURRENT_DATE(),10) AS plus10,
DATE_SUB(CURRENT_DATE(),10) AS minus10;

SELECT
emp_name,
DATEDIFF(CURRENT_DATE(), joining_date) AS days_worked
FROM employees;

-- Joined in last 30 days

SELECT *
FROM employees
WHERE joining_date >= DATE_SUB(CURRENT_DATE(),30);

-- ============================================
-- 7. STRING FUNCTIONS
-- ============================================

SELECT
emp_name,
UPPER(emp_name),
LOWER(emp_name),
LENGTH(emp_name),
CONCAT(emp_name,' Works'),
SUBSTRING(emp_name,1,3),
TRIM('   Databricks   ')
FROM employees;

-- ============================================
-- 8. WINDOW FUNCTIONS
-- ============================================

SELECT
emp_name,
salary,
ROW_NUMBER() OVER(ORDER BY salary DESC) rn,
RANK() OVER(ORDER BY salary DESC) rnk,
DENSE_RANK() OVER(ORDER BY salary DESC) drnk
FROM employees;

-- Running Total

SELECT
emp_name,
salary,
SUM(salary)
OVER(ORDER BY emp_id)
AS running_total
FROM employees;

-- ============================================
-- 9. DEDUPLICATION
-- ============================================

CREATE OR REPLACE TABLE duplicate_emp
USING DELTA
AS
SELECT * FROM employees;

INSERT INTO duplicate_emp
SELECT * FROM employees WHERE emp_id=101;

SELECT * FROM duplicate_emp;

SELECT *
FROM
(
SELECT *,
ROW_NUMBER() OVER(PARTITION BY emp_id ORDER BY emp_id) rn
FROM duplicate_emp
)
WHERE rn=1;

-- ============================================
-- 10. PIVOT
-- ============================================

SELECT *
FROM
(
SELECT department,salary
FROM employees
)
PIVOT
(
SUM(salary)
FOR department IN ('IT','HR','Finance','Sales')
);

-- ============================================
-- 11. QUALIFY
-- ============================================

SELECT
emp_name,
department,
salary
FROM employees
QUALIFY ROW_NUMBER()
OVER(PARTITION BY department ORDER BY salary DESC)=1;

-- ============================================
-- 12. STATISTICS
-- ============================================

ANALYZE TABLE employees COMPUTE STATISTICS;

ANALYZE TABLE employees
COMPUTE STATISTICS
FOR COLUMNS department,salary;

-- ============================================
-- 13. DELTA MERGE
-- ============================================

CREATE OR REPLACE TABLE employee_updates
(
emp_id INT,
emp_name STRING,
department STRING,
salary INT,
joining_date DATE
)
USING DELTA;

INSERT INTO employee_updates VALUES
(102,'Rahul','HR',70000,'2026-06-15'),
(106,'Kiran','IT',52000,'2026-07-19');

MERGE INTO employees e
USING employee_updates u
ON e.emp_id=u.emp_id

WHEN MATCHED THEN
UPDATE SET
e.salary=u.salary

WHEN NOT MATCHED THEN
INSERT *;

SELECT * FROM employees;

-- ============================================
-- 14. UPDATE
-- ============================================

UPDATE employees
SET salary=75000
WHERE emp_id=101;

SELECT * FROM employees;

-- ============================================
-- 15. DELETE
-- ============================================

DELETE FROM employees
WHERE emp_id=105;

SELECT * FROM employees;

-- ============================================
-- 16. OPTIMIZE
-- ============================================

OPTIMIZE employees;

-- ============================================
-- 17. TIME TRAVEL
-- ============================================

DESCRIBE HISTORY employees;

-- Replace 0 with an actual version from DESCRIBE HISTORY
SELECT *
FROM employees
VERSION AS OF 0;

-- ============================================
-- 18. VACUUM
-- ============================================

VACUUM employees RETAIN 168 HOURS;

-- ============================================
-- 19. SCHEMA EVOLUTION
-- ============================================

ALTER TABLE employees
ADD COLUMNS (email STRING);

DESCRIBE employees;

-- ============================================
-- 20. QUERY PROFILE
-- ============================================

SELECT *
FROM employees
WHERE department='IT';

-- After running:
-- Click Query Profile in Databricks UI
